<a href="https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/ColabFold2_preview.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ColabFold2 preview

Predict protein, RNA, DNA and small-molecule structures with
[AlphaFold 3](https://www.nature.com/articles/s41586-024-07487-w) — and with thirteen
other sets of weights, all through one implementation. Pick a model in the install
cell; the weights download themselves. MSAs come from the
[ColabFold](https://github.com/sokrypton/ColabFold) MMseqs2 server, so no local
databases are needed, and the attention/XLA flags are chosen for whatever GPU you get.

**Models, licences, download sizes and every option: the Instructions cell at the
bottom.**

**Please cite** AlphaFold 3 ([Abramson 2024](https://doi.org/10.1038/s41586-024-07487-w)),
ColabFold ([Mirdita 2022](https://doi.org/10.1038/s41592-022-01488-1)), and whichever
model you ran.


In [ ]:
#@title Install dependencies (~35 s)
# No `%%time` here: a cell magic must be the first line and `#@title` already is.
import os, time, glob, shutil, sys
_T0 = time.time()

model = "openbind0" #@param ["openbind0", "openfold3", "boltz2", "protenix2", "rosettafold3", "chai1", "intellifold2", "opendde", "esmfold2", "esmfold2_lm600m", "esmfold2_lm300m", "alphafold3", "af2_ptm", "af2_multimer"]

persist_cache_to_drive = False #@param {type:"boolean"}
#@markdown - **persist_cache_to_drive**: keep the compiled model in Drive so the next
#@markdown   session skips the recompile -- worth ~53 s (69 s cold vs 16 s warm on a
#@markdown   68-residue input). Never changes a result.

# Form fields are plain assignments, so a headless run (colab-cli, CI) sets them
# through the environment: AF3_NB_OVERRIDES='{"model": "boltz2"}'
import json as _json
for _k, _v in _json.loads(os.environ.get('AF3_NB_OVERRIDES', '{}')).items():
  if _k in globals():
    globals()[_k] = _v
    print(f'override: {_k} = {_v!r}')

VERSION = '3.1.10'          # package and run_alphafold.py both come from this tag
NATIVE_DIR = 'af3_native_weights'
AF2_DIR = 'af2_params'
IS_AF3 = (model == 'alphafold3')
IS_AF2 = model.startswith('af2_')
# int8: the same weights stored 8-bit and expanded on load, which is what keeps
# the download to a few hundred MB. AF2 and AF3 ship their own float32 files.
PRECISION = 'fp32' if (IS_AF3 or IS_AF2) else 'int8'


def _sh(cmd, what):
  """Run a shell command and raise if it fails -- os.system's status is easy to drop."""
  if os.system(cmd) != 0:
    raise RuntimeError(f'{what} failed. The output is above.')


if not os.path.isfile('ALPHAFOLD3_READY'):
  print('Installing packages...')
  # --no-deps throughout: Colab already ships jax and the CUDA stack, and letting
  # pip re-resolve them re-downloads gigabytes. So every third-party import the
  # package needs is listed here instead.
  _sh("pip install -q dm-haiku==0.0.17 rdkit==2025.9.4 "
      "tokamax==0.0.11 ml_collections", 'installing dependencies')
  _sh("pip install -q git+https://github.com/sokrypton/py2Dmol.git",  # wheel lags the repo
      'installing py2Dmol')
  if IS_AF2:
    os.system("apt-get -qq install -y aria2 > /dev/null 2>&1")  # AF2's tar is 5.3 GB
  # Retried: a wheel published minutes ago may not be in PyPI's index yet.
  for _try in range(4):
    if os.system(f'pip install -q --no-deps alphafold3-colabfold=={VERSION}') == 0:
      break
    print(f'pip could not find {VERSION} yet; retrying in 20 s')
    time.sleep(20)
  else:
    raise RuntimeError(f'could not install alphafold3-colabfold=={VERSION}')
  # run_alphafold.py is a top-level script, not part of the package.
  _sh(f'wget -q -O run_alphafold.py https://raw.githubusercontent.com'
      f'/sokrypton/alphafold3/v{VERSION}/run_alphafold.py', 'fetching run_alphafold.py')
  # haiku 0.0.17 still calls the moved jax.core.DropVar.
  os.system("sed -i 's/jax.core.DropVar/jax.extend.core.DropVar/g' /usr/local/lib/python*/dist-packages/haiku/_src/jaxpr_info.py")
  import alphafold3  # the only real proof the install worked
  os.system('touch ALPHAFOLD3_READY')
  print(f'Packages installed ({alphafold3.__file__}).')

# tokamax enables its Triton kernels for every GPU with cc >= 8.0, but they need
# more shared memory than Ada cards have and fail at launch. Restrict them to
# datacenter GPUs (A100 cc 8.0, H100 cc 9.0+); Ada/L4 use XLA like a T4 does.
try:
  import tokamax
  _gu = os.path.join(os.path.dirname(tokamax.__file__), '_src', 'gpu_utils.py')
  _s = open(_gu).read()
  _old = 'return float(device.compute_capability) >= 8.0'
  _new = ('cc = float(device.compute_capability)\n'
          '  return cc == 8.0 or cc >= 9.0  # datacenter only; Ada/L4 lack shared memory')
  if _old in _s:
    open(_gu, 'w').write(_s.replace(_old, _new))
    print('Patched tokamax: Triton restricted to datacenter GPUs (L4/Ada -> XLA).')
except Exception as _e:
  print(f'(tokamax patch skipped: {_e})')

# Weights, fetched in the background by the same code the run uses, so the cache
# layout cannot drift between the two.
STAMP = f'WEIGHTS_DONE_{model}_{PRECISION}'
if IS_AF3:
  # NOT downloaded. DeepMind's AlphaFold 3 parameters are granted on request,
  # for non-commercial research, at Google's discretion -- they are not ours to
  # fetch on your behalf. Apply at
  # https://docs.google.com/forms/d/e/1FAIpQLSfWZAgo1aYk0O4MuAXZj8xRQ8DafeFJnldNOnh_13qAx2ceZw/viewform
  # and put the file you are given in af3_native_weights/.
  os.makedirs(NATIVE_DIR, exist_ok=True)
  _blobs = glob.glob(f'{NATIVE_DIR}/*.bin.zst')
  if not _blobs:
    raise RuntimeError(
        f'No AlphaFold 3 parameters found in {NATIVE_DIR}/.\n'
        'DeepMind grants these on request (non-commercial research); this '
        'notebook cannot download them for you. Request them at\n'
        '  https://github.com/google-deepmind/alphafold3#obtaining-model-parameters\n'
        f'then upload the .bin.zst file into {NATIVE_DIR}/ and re-run this cell.\n'
        'Every other model in the dropdown downloads its own weights.')
  print(f'Using your AlphaFold 3 parameters: {_blobs[0]}')
elif not os.path.isfile(STAMP):
  _script, _args = ('prefetch_af2.py', AF2_DIR) if IS_AF2 else (
      'prefetch_weights.py', f'{model} {PRECISION}')
  print(f'Downloading {"official AlphaFold 2 parameters (CC BY 4.0)" if IS_AF2 else model} weights...')
  with open(_script, 'w') as fh:
    fh.write('import sys\n'
             'from alphafold3.model import weights\n'
             + ('print(weights.ensure_af2_params(sys.argv[1]))\n' if IS_AF2 else
                'print(weights.ensure_weights(sys.argv[1], None, precision=sys.argv[2]))\n'))
  os.system(f'(python {_script} {_args} > {STAMP}.log 2>&1 && touch {STAMP}) &')

# /tmp is wiped with the VM, so a fresh session recompiles (~53 s on a small
# input); Drive survives. Falls back to /tmp rather than failing.
CACHE_DIR = '/tmp/af3_cache'
if persist_cache_to_drive:
  try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = '/content/drive/MyDrive/.af3_cache'
    os.makedirs(CACHE_DIR, exist_ok=True)
    print(f'Compile cache: {CACHE_DIR} (survives this session)')
  except Exception as _e:
    print(f'(Drive mount failed, using {CACHE_DIR}: {_e})')


def _await(sentinel, limit=1200):
  """Wait for a background job, with a bound -- a failed download must not hang."""
  t0 = time.time()
  while not os.path.isfile(sentinel):
    if time.time() - t0 > limit:
      log = f'{sentinel}.log'
      tail = open(log).read()[-1500:] if os.path.isfile(log) else '(no output captured)'
      raise RuntimeError(f'{sentinel} did not appear within {limit} s. '
                         f'Tail of {log}:\n{tail}')
    time.sleep(5)
  print(f'{sentinel} \u2713  ({time.time() - t0:.0f} s)')


if not IS_AF3:
  _await(STAMP)

print(f'Setup complete!  Model: {model}.')
if model == 'chai1':
  print('NOTE: chai-1 is running WITHOUT ESM2 embeddings, which are most of its token\n'
        '      features. Expect worse structures than chai-lab itself produces.')
print(f'Setup took {time.time() - _T0:.0f} s.')


In [ ]:
#@title Input sequences
import re, os, json, hashlib

#@markdown ### Molecules
#@markdown Separate multiple chains within a box using `:` (extra colons are fine: `A::::B` == `A:B`). Leave a box empty if unused; full details in the Instructions cell.
protein = 'PIAQIHILEGRSDEQKETLIREVSEAISRSLDAPLTSVRVIITEMAKGHFGIGGELASK' #@param {type:"string"}
dna = '' #@param {type:"string"}
rna = '' #@param {type:"string"}
ligand_ccd = '' #@param {type:"string"}
ligand_smiles = '' #@param {type:"string"}

#@markdown ### Run settings
jobname = 'test' #@param {type:"string"}
msa_mode = "mmseqs2_server" #@param ["mmseqs2_server", "single_sequence"]
seeds = '1' #@param {type:"string"}
on_existing = "overwrite" #@param ["overwrite", "skip"]
#@markdown - `msa_mode`: `single_sequence` skips the MSA (faster, lower accuracy).
#@markdown - `seeds`: comma-separated, e.g. `1,2,3`.
#@markdown - `on_existing`: `overwrite` replaces this job's previous results; `skip` keeps them.

# Headless overrides -- see the install cell.
import json as _json, os as _os
for _k, _v in _json.loads(_os.environ.get('AF3_NB_OVERRIDES', '{}')).items():
  if _k in globals():
    globals()[_k] = _v
    print(f'override: {_k} = {_v!r}')

# Split a box into entries: collapse colon runs, drop whitespace, skip empties
def split_entries(s):
  s = re.sub(r':+', ':', s).strip(':')
  return [e for e in (''.join(tok.split()) for tok in s.split(':')) if e]

prot_seqs   = [e.upper() for e in split_entries(protein)]
dna_seqs    = [e.upper() for e in split_entries(dna)]
rna_seqs    = [e.upper() for e in split_entries(rna)]
ccd_codes   = [e.upper() for e in split_entries(ligand_ccd)]
smiles_strs = split_entries(ligand_smiles)         # case-sensitive: leave as typed

# The CCD, for the components this input names -- which is why it is here and
# not in the install cell, which runs before you have typed a ligand. Each one
# comes from files.rcsb.org (kilobytes, ~0.6 s) instead of build_data parsing
# libcifpp's whole 518 MB dictionary. A code that was not fetched raises.
with open('prefetch_ccd.py', 'w') as fh:
  fh.write('import sys, os, importlib.metadata as md\n'
           'from alphafold3.constants import ccd_fetch\n'
           'root = os.path.dirname(md.distribution("alphafold3-colabfold")'
           '.locate_file("alphafold3"))\n'
           'conv = os.path.join(root, "alphafold3", "constants", "converters")\n'
           'os.makedirs(conv, exist_ok=True)\n'
           'ccd_fetch.write_pickles(ccd_fetch.codes_for_input(extra=sys.argv[1:]),\n'
           '  os.path.join(conv, "ccd.pickle"),\n'
           '  os.path.join(conv, "chemical_component_sets.pickle"),\n'
           '  libcifpp_dir=os.path.join(root, "share", "libcifpp"))\n')
print(f'Fetching the CCD: 35 standard residues'
      + (f' + {", ".join(ccd_codes)}' if ccd_codes else '') + ' ...')
if os.system('python prefetch_ccd.py ' + ' '.join(ccd_codes)) != 0:
  raise RuntimeError('could not build the CCD tables; see the output above')

# Build AF3 chain entities (IDs A, B, C, ... in canonical order)
CHAIN_IDS = list('ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz')
chains, prot_groups, idx = [], {}, 0

for seq in prot_seqs:
  cid = CHAIN_IDS[idx]; idx += 1
  if seq in prot_groups:                           # merge identical seqs -> homo-oligomer
    ent = prot_groups[seq]
    ids = ent['id'] if isinstance(ent['id'], list) else [ent['id']]
    ent['id'] = ids + [cid]
  else:
    ent = {'id': cid, 'sequence': seq, 'templates': []}
    if msa_mode == 'single_sequence':
      ent.update({'unpairedMsa': f'>query\n{seq}\n', 'pairedMsa': ''})
    prot_groups[seq] = ent
    chains.append({'protein': ent})

for seq in rna_seqs:
  c = {'id': CHAIN_IDS[idx], 'sequence': seq}
  if msa_mode == 'single_sequence':
    c['unpairedMsa'] = f'>query\n{seq}\n'
  chains.append({'rna': c}); idx += 1

for seq in dna_seqs:
  chains.append({'dna': {'id': CHAIN_IDS[idx], 'sequence': seq}}); idx += 1

for code in ccd_codes:
  chains.append({'ligand': {'id': CHAIN_IDS[idx], 'ccdCodes': [code]}}); idx += 1

for smiles in smiles_strs:
  chains.append({'ligand': {'id': CHAIN_IDS[idx], 'smiles': smiles}}); idx += 1

if not chains:
  raise ValueError('No valid input found - fill in at least one box.')

# Seeds: pull out integers regardless of separators, dedupe, default to [1]
seed_list = []
for tok in re.findall(r'\d+', seeds):
  v = int(tok)
  if v not in seed_list:
    seed_list.append(v)
if not seed_list:
  seed_list = [1]

# Deterministic, lower-cased job name from inputs+seeds.
# Same input+seeds -> same folder (so re-runs reuse it instead of piling up).
# Lower-cased to match run_alphafold.py's sanitised_name() output directory.
def _flat(mol):
  if 'sequence' in mol: return mol['sequence']
  if 'ccdCodes' in mol: return ','.join(mol['ccdCodes'])
  return mol.get('smiles', '?')
flat = ':'.join(_flat(list(c.values())[0]) for c in chains) + '|seeds=' + ','.join(map(str, seed_list))
basejob = (re.sub(r'\W+', '', ''.join(jobname.split())) or 'job').lower()
jobname = basejob + '_' + hashlib.sha1(flat.encode()).hexdigest()[:5]

# Input JSON goes to a temp dir; ALL results land in ONE folder: af3_output/<jobname>/
INPUT_DIR  = '/tmp/af3_inputs'
OUTPUT_DIR = 'af3_output'
job_dir    = f'{OUTPUT_DIR}/{jobname}'

fold_input = {
    'name': jobname,
    'sequences': chains,
    'modelSeeds': seed_list,
    'dialect': 'alphafold3',
    'version': 1,
}
os.makedirs(INPUT_DIR, exist_ok=True)
json_path = f'{INPUT_DIR}/{jobname}.json'
with open(json_path, 'w') as f:
  json.dump(fold_input, f, indent=2)

print(f'Job "{jobname}"  ->  results will be written to {job_dir}/')
fold_input


In [ ]:
#@title Run the model
# No `%%time` -- see the install cell.
import os, shutil, subprocess, glob, time
_T0 = time.time()

#@markdown Defaults match AlphaFold 3; raise only if needed.
num_recycles = 10 #@param {type:"integer"}
num_diffusion_samples = 5 #@param {type:"integer"}
#@markdown - `num_recycles`: refinement passes; more helps hard targets, costs time.
#@markdown - `num_diffusion_samples`: structures per seed, so total = seeds x samples.

# Headless overrides -- see the install cell.
import json as _json, os as _os
for _k, _v in _json.loads(_os.environ.get('AF3_NB_OVERRIDES', '{}')).items():
  if _k in globals():
    globals()[_k] = _v
    print(f'override: {_k} = {_v!r}')

num_recycles = max(1, int(num_recycles))
num_diffusion_samples = max(1, int(num_diffusion_samples))

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Re-run policy (one folder per job, no timestamped duplicates):
#   overwrite -> wipe this job's folder and recompute
#   skip      -> if a finished result (.cif) is already there, don't recompute
have_results = os.path.isdir(job_dir) and any(f.endswith('.cif') for f in os.listdir(job_dir))
run_it = not (on_existing == 'skip' and have_results)
if run_it:
  shutil.rmtree(job_dir, ignore_errors=True)   # start clean so exactly one folder is produced

# Pick attention impl + XLA flags from the actual device.
# Triton/cuDNN flash attention need Ampere (compute capability >= 8.0);
# 7.x GPUs (T4=7.5, V100=7.0) and CPU use the portable XLA path, and 7.x
# additionally needs the XLA flag that disables the custom-kernel fusion pass.
def detect_device():
  try:
    out = subprocess.run(
        ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=15)
    caps = [float(x) for x in out.stdout.split() if x.strip()]
    if caps:
      return 'gpu', min(caps)
  except Exception:
    pass
  return 'cpu', None

device, cap = detect_device()
nojit = False
xla_flags = []   # extra XLA flags to export for this device (per AlphaFold 3's guidance)

if device == 'cpu':
  flash_impl = 'xla'
  nojit = True
  print('No GPU detected - running on CPU with XLA attention + --nojit (slow, but avoids the compile).')
elif cap < 8.0:
  # T4 / V100 (cc 7.x): XLA attention; disable the custom-kernel fusion pass.
  # (Triton GEMM is not supported on these cards, so it is not disabled here.)
  flash_impl = 'xla'
  xla_flags = ['--xla_disable_hlo_passes=custom-kernel-fusion-rewriter']
  print(f'Pre-Ampere GPU (compute capability {cap}) - XLA attention + custom-kernel fusion disabled.')
elif 8.0 < cap < 9.0:
  # L4 / Ada / consumer Ampere (cc 8.6 / 8.9): limited shared memory. XLA's Triton GEMM
  # kernels exceed it ('Shared memory size limit exceeded'), so disable Triton GEMM
  # (falls back to cuBLAS) and use XLA attention to also avoid the Triton attention kernel.
  flash_impl = 'xla'
  xla_flags = ['--xla_gpu_enable_triton_gemm=false']
  print(f'Ada/consumer GPU (compute capability {cap}) - XLA attention + Triton GEMM disabled (shared-memory limit).')
else:
  # A100 (cc 8.0) and H100 (cc 9.0+): ample shared memory. Triton flash attention,
  # with Triton GEMM disabled per AlphaFold 3's recommended XLA_FLAGS.
  flash_impl = 'triton'
  xla_flags = ['--xla_gpu_enable_triton_gemm=false']
  print(f'Datacenter GPU (compute capability {cap}) - Triton flash attention + Triton GEMM disabled.')

# Export XLA flags so the child shell (and JAX inside it) inherit them.
cur = os.environ.get('XLA_FLAGS', '')
for f in xla_flags:
  if f not in cur:
    cur = (cur + ' ' + f).strip()
if cur:
  os.environ['XLA_FLAGS'] = cur

print('XLA_FLAGS =', os.environ.get('XLA_FLAGS', '(unset)'))

# Weights. Every ported model resolves its own cache directory (populated by the
# install cell), so --model_dir is passed only for the two whose parameters come
# from DeepMind directly: AlphaFold 3's, and AlphaFold 2's (CC BY 4.0, fetched
# into af2_params by the install cell).
print(f'Model: {model}')

cmd = [
    'python', 'run_alphafold.py',
    f'--json_path={json_path}',
    f'--model={model}',
    '--norun_data_pipeline',
    f'--output_dir={OUTPUT_DIR}',
    f'--cache_dir={CACHE_DIR}',
    '--force_output_dir',          # reuse af3_output/<jobname>/ instead of a timestamped copy
    f'--flash_attention_implementation={flash_impl}',
    f'--num_recycles={num_recycles}',
    f'--num_diffusion_samples={num_diffusion_samples}',
]
if msa_mode == 'mmseqs2_server':
  cmd.append('--use_msa_server')
# chai-1 folds from ESM2 and ESMFold2 from ESM-C; without it they are a
# different model, not a slightly worse one (a natural protein goes to 5.70 A
# where chai-1 reaches 0.642, and an ESMFold2 variant with no MSA encoder has
# nothing left to fold from at all). Both towers run in-process and download on
# demand, which is why run_alphafold makes it opt-in and this passes it.
if model == 'chai1' or model.startswith('esmfold2'):
  cmd.append('--use_esm_embeddings')
if nojit:
  cmd.append('--nojit')
if IS_AF3 or IS_AF2:
  cmd.append(f'--model_dir={AF2_DIR if IS_AF2 else NATIVE_DIR}')

cmd = ' '.join(cmd)
if run_it:
  print(cmd)
  # Popen, not `!{cmd}`: it streams the same output but also yields a status,
  # so a failed run cannot report `Done ->`.
  _p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
  for _line in _p.stdout:
    print(_line, end='')
  _rc = _p.wait()
  _cifs = glob.glob(f'{job_dir}/**/*.cif', recursive=True)
  if _rc != 0 or not _cifs:
    raise RuntimeError(
        f'the fold FAILED (exit {_rc}, {len(_cifs)} structures written). '
        'The output above is the whole story; scroll up for the error.')
  print(f'\nDone -> {job_dir}/  ({len(_cifs)} structures, '
        f'{time.time() - _T0:.0f} s)')
else:
  print(f'Skipping: results already exist in {job_dir}/  (set on_existing=overwrite to recompute).')


In [ ]:
#@title Display structures + PAE (py2Dmol)
import csv, glob, os, json
import numpy as np
import py2Dmol

load_as_frames = True #@param {type:"boolean"}
viewer_size = 400
#@markdown All predicted models load together, best first (**rank_1, rank_2, ...**), each with its own PAE.
#@markdown - `load_as_frames` **off** -> pick a model from the dropdown.
#@markdown - `load_as_frames` **on**  -> models become frames you can play through (press play / drag the slider).
#@markdown - The interactive PAE matrix sits beside the structure; click or drag-box on it to highlight residues.

# All models in rank order (best first): from the ranking CSV, fall back to globbing.
def collect_models():
  ranking_csv = f'{job_dir}/{jobname}_ranking_scores.csv'
  cifs = []
  if os.path.exists(ranking_csv):
    rows = []
    with open(ranking_csv) as f:
      for r in csv.DictReader(f):
        rows.append((float(r['ranking_score']), int(r['seed']), int(r['sample'])))
    for _, seed, sample in sorted(rows, reverse=True):
      d = f'{job_dir}/seed-{seed}_sample-{sample}'
      hit = sorted(glob.glob(f'{d}/*_model.cif')) or sorted(glob.glob(f'{d}/*.cif'))
      if hit:
        cifs.append(hit[0])
  if not cifs:
    cifs = (sorted(glob.glob(f'{job_dir}/**/*_model.cif', recursive=True))
            or sorted(glob.glob(f'{job_dir}/**/*.cif', recursive=True)))
  return cifs

# Per-model PAE: confidences.json next to the CIF, else the top-level one.
def load_pae(cif):
  d = os.path.dirname(cif)
  cands = [p for p in glob.glob(f'{d}/*_confidences.json')
           if 'summary' not in os.path.basename(p)]
  if not cands:
    top = f'{job_dir}/{jobname}_confidences.json'
    cands = [top] if os.path.exists(top) else []
  if cands:
    pae = json.load(open(cands[0])).get('pae')
    if pae is not None:
      return np.asarray(pae, dtype=float)
  return None

cifs = collect_models()
if not cifs:
  raise FileNotFoundError(f'No model CIFs found in {job_dir}/')
print(f'Loaded {len(cifs)} model(s) from {job_dir}/'
      + ('  (frames - press play)' if load_as_frames else '  (use the dropdown to switch)'))

viewer = py2Dmol.view(size=(viewer_size, viewer_size),
                      pae=True, autoplay=load_as_frames,
                      style="cartoon")
for i, cif in enumerate(cifs, start=1):
  pae = load_pae(cif)
  if load_as_frames:
    viewer.add_pdb(cif, name='models', paes=pae)      # same name -> frames (play through)
  else:
    viewer.add_pdb(cif, name=f'rank_{i}', paes=pae)    # distinct names -> dropdown of objects
viewer.show()


In [ ]:
#@title Quality metrics and plots
import json, os
import numpy as np
import matplotlib.pyplot as plt

conf_path = f'{OUTPUT_DIR}/{jobname}/{jobname}_confidences.json'
summ_path = f'{OUTPUT_DIR}/{jobname}/{jobname}_summary_confidences.json'

with open(conf_path) as f:
  conf = json.load(f)
with open(summ_path) as f:
  summ = json.load(f)

plddts = np.array(conf.get('atom_plddts', conf.get('token_plddts', [])), dtype=float)
plddt_chain_ids = conf.get('atom_chain_ids', conf.get('token_chain_ids', []))   # pLDDT is per-ATOM
token_chain_ids = conf.get('token_chain_ids', [])                                # PAE is per-TOKEN
pae = np.array(conf.get('pae', []), dtype=float)

# ── Summary (ipTM is None for single-chain jobs — guard before formatting) ─
def fmt(v):
  return f'{v:.3f}' if isinstance(v, (int, float)) else 'n/a'

mean_plddt = summ.get('mean_plddt')
if mean_plddt is None and plddts.size:
  mean_plddt = float(np.mean(plddts))
iptm = summ.get('iptm')

print('=' * 38)
print(f'Mean pLDDT     : {fmt(mean_plddt)}')
print(f'pTM            : {fmt(summ.get("ptm"))}')
print(f'ipTM           : {fmt(iptm)}' + ('   (single chain — no interface)' if iptm is None else ''))
print(f'Ranking score  : {fmt(summ.get("ranking_score"))}')
print('=' * 38)

# ── Plots ───────────────────────────────────────────────────
has_pae = pae.ndim == 2 and pae.size > 0
ncols = 2 if has_pae else 1
fig, axes = plt.subplots(1, ncols, figsize=(13 if has_pae else 6.5, 4))
axes = np.atleast_1d(axes)

# pLDDT per residue — a line (coloured per chain when there is more than one)
ax = axes[0]
x = np.arange(len(plddts))
xmax = max(len(plddts) - 1, 1)
ax.set_xlim(0, xmax)
ax.set_ylim(0, 100)

unique_chains = list(dict.fromkeys(plddt_chain_ids))
if len(unique_chains) > 1:
  colors = plt.cm.tab10(np.linspace(0, 1, len(unique_chains)))
  tcid = np.array(plddt_chain_ids)
  for ch, col in zip(unique_chains, colors):
    y = np.where(tcid == ch, plddts, np.nan)   # NaN gaps keep chains as separate lines
    ax.plot(x, y, lw=1.5, color=col, label=f'Chain {ch}')
  for b in [i for i in range(1, len(plddt_chain_ids)) if plddt_chain_ids[i] != plddt_chain_ids[i-1]]:
    ax.axvline(b - 0.5, color='grey', lw=0.6, alpha=0.5)
  ax.legend(loc='lower right', fontsize=8)
else:
  ax.plot(x, plddts, lw=1.5, color='#1f77b4')

for y in (50, 70, 90):
  ax.axhline(y, ls='--', lw=0.7, color='grey', alpha=0.5)
  ax.text(xmax, y, f' {y}', va='center', ha='left', fontsize=7, color='grey')
ax.set_xlabel('Atom')
ax.set_ylabel('pLDDT')
ax.set_title('Predicted pLDDT per atom')

# PAE matrix
if has_pae:
  ax = axes[1]
  im = ax.imshow(pae, cmap='bwr', vmin=0, vmax=30, interpolation='nearest')
  plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='PAE (Å)')
  if token_chain_ids:
    for b in [i for i in range(1, len(token_chain_ids)) if token_chain_ids[i] != token_chain_ids[i-1]]:
      ax.axhline(b - 0.5, c='black', lw=0.8)
      ax.axvline(b - 0.5, c='black', lw=0.8)
  ax.set_xlabel('Scored residue')
  ax.set_ylabel('Aligned residue')
  ax.set_title('Predicted Aligned Error (PAE)')

plt.tight_layout()
plt.show()


In [ ]:
#@title Download results
from google.colab import files
import os

results_zip = f'{jobname}.result.zip'
os.system(f'zip -r {results_zip} {OUTPUT_DIR}/{jobname}')
files.download(results_zip)


# Instructions <a name="Instructions"></a>

**Quick start:** pick a **model** in the install cell, fill in the sequence(s), then
**Runtime → Run all**. The first run downloads that model's weights; later runs reuse
them, and each model has its own cache so switching back is instant.

---

## Models

Ported weights come from [sokrypton/af3-any-model](https://huggingface.co/sokrypton/af3-any-model).
"Tower" is a protein language model fetched separately on first use.

| model | weights | licence | notes |
|---|---|---|---|
| `openbind0` | [OpenFold3 v0.5.0 "OpenBind"](https://github.com/aqlaboratory/openfold-3/releases/tag/v0.5.0) (AlQuraishi Lab) | Apache-2.0 | The current release, and the default here. |
| `openfold3` | [OpenFold3 preview-2](https://github.com/aqlaboratory/openfold) (AlQuraishi Lab) | Apache-2.0 | The earlier preview, kept because earlier results used it. |
| `boltz2` | [Boltz-2](https://github.com/jwohlwend/boltz) (Wohlwend et al.) | MIT | Strong on ligands; keeps a modified residue as one token. |
| `protenix2` | [Protenix-v2](https://github.com/bytedance/Protenix) (ByteDance) | Apache-2.0 | The widest trunk here (pair 256), so the slowest. |
| `rosettafold3` | [RoseTTAFold3](https://github.com/RosettaCommons/foundry) (RosettaCommons) | BSD-3-Clause | Carries chirality features; handles D-amino acids. |
| `chai1` | [chai-1](https://github.com/chaidiscovery/chai-lab) (Chai Discovery) | Apache-2.0 | Folds from ESM2 3B, fetched and run automatically. |
| `intellifold2` | [IntelliFold-v2](https://huggingface.co/intelligenAI/intellifold) (IntelliGen-AI) | Apache-2.0 | Widened channels (pair 512), largest ported download. |
| `opendde` | [OpenDDE](https://huggingface.co/aurekaresearch/OpenDDE) (Aureka Research) | Apache-2.0 | Runs its diffusion on an expanded structural-token set. |
| `esmfold2` | [ESMFold2](https://huggingface.co/biohub/ESMFold2) (Chan Zuckerberg Biohub) | MIT | Folds from ESM-C instead of an MSA — single sequence, no search. |
| `esmfold2_lm600m` | ESMFold2, 600M tower | MIT | No confidence head. |
| `esmfold2_lm300m` | ESMFold2, 300M tower | MIT | No confidence head. |
| `af2_ptm` | AlphaFold 2 monomer pTM (DeepMind) | CC BY 4.0 | **Protein only** — a ligand or nucleotide raises rather than quietly folding the rest. Templates use the model_1/model_2 parameter sets. |
| `af2_multimer` | AlphaFold 2 multimer v3 (DeepMind) | CC BY 4.0 | Protein only, as above. |
| `alphafold3` | Google DeepMind's own parameters | [AF3 terms of use](https://github.com/google-deepmind/alphafold3/blob/main/WEIGHTS_TERMS_OF_USE.md) | Requires requesting the weights from Google (non-commercial research only, granted at Google's discretion) — not a direct download. run_alphafold prints the terms at startup. |
---

## Input

Each molecule type has its own box; within a box, separate chains with `:`.

| box | contents | example |
|---|---|---|
| **protein** | amino-acid sequence(s) | `MKTAY...` or `SEQ1:SEQ2` |
| **dna** | DNA sequence(s) | `CGCGAATTCGCG` |
| **rna** | RNA sequence(s) | `GCGGAUUUA` |
| **ligand_ccd** | ligand(s) by PDB CCD code | `ATP:MG:HEM` |
| **ligand_smiles** | ligand(s) by SMILES | `CC(=O)Oc1ccccc1C(=O)O` |

Mix boxes freely to build a complex. Chain IDs A, B, C… follow AlphaFold 3's canonical
order (protein → RNA → DNA → ligand). Identical protein sequences are merged, so
`SEQ:SEQ` is a homodimer. Sequences and CCD codes are upper-cased; **SMILES are left
as typed**. Whitespace and extra colons are forgiven (`SEQ1::::SEQ2` = `SEQ1:SEQ2`) —
which is also why an atom-mapped SMILES containing `:` needs a raw AF3 JSON instead.

**seeds**: comma-separated, one prediction each (`1,2,3`). Junk and duplicates are
dropped. **msa_mode**: `mmseqs2_server` queries the public
[ColabFold](https://colabfold.mmseqs.com/) API (protein only — RNA/DNA always run
MSA-free); `single_sequence` skips it, faster and less accurate.

## Output

| file | contents |
|---|---|
| `*.cif` | Best-ranked structure. B-factor = pLDDT (0–100). |
| `*_confidences.json` | Per-residue pLDDT, PAE matrix, contact probabilities. |
| `*_summary_confidences.json` | Mean pLDDT, pTM, ipTM, ranking score. |
| `*_ranking_scores.csv` | Every seed × sample combination. |
| `seed-N_sample-M/` | One directory per prediction. |
| `TERMS_OF_USE.md` | The licence for whichever weights you ran. |

pLDDT above 90 is very high, 70–90 reliable backbone, 50–70 doubtful, below 50 likely
disordered or wrong. Lower PAE means two residues are confidently placed *relative to
each other*, which is what to read for an interface. ipTM above 0.8 is a well-defined
complex interface, and is `n/a` for a single chain — there is no interface to score.

## Troubleshooting

**OOM**: shorter sequence, or a larger GPU (`Runtime → Change runtime type`).
**MSA server timeout**: the public server is rate-limited — retry, or use
`single_sequence`. **Download popup blocked**: disable your ad blocker.

## Licence

The AlphaFold 3 **source code** is [Apache 2.0](https://www.apache.org/licenses/LICENSE-2.0);
the **weights** are each their own, as listed in the model table. Outputs from the seven
Apache/MIT/BSD-licensed ported models are **not** subject to DeepMind's AlphaFold 3 Output
Terms of Use and may be used freely, including commercially. `alphafold3` is the exception:
its parameters and outputs carry DeepMind's own terms. Every run writes a
`TERMS_OF_USE.md` naming the licence that actually applies to it.

## Bugs / feedback

https://github.com/sokrypton/alphafold3/issues
